# Al-Nisyan MeKi Hybrid Notebook
Run this on Kaggle (GPU T4) to evaluate embedding-level hybrid memory injection.

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth transformers accelerate bitsandbytes torch

In [ ]:
# Cell 2: Clone repo
import os, sys, subprocess
from pathlib import Path

repo_url = "https://github.com/faresrafat3/al-nisyan.git"
repo_path = Path('/kaggle/working/al-nisyan')

if repo_path.exists():
    subprocess.run(['rm', '-rf', str(repo_path)], check=True)

subprocess.run(['git', 'clone', repo_url, str(repo_path)], check=True)
os.chdir(repo_path)
sys.path.insert(0, str(repo_path))
print('Repo ready:', repo_path)

In [ ]:
# Cell 3: GPU check
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
# Cell 4: Load base model
from src.models.gemma_loader import GemmaLoader

loader = GemmaLoader(model_key='Qwen/Qwen3.5-4B')
model, tokenizer = loader.load(load_in_4bit=True)
print('Model loaded:', loader.get_model_info())
if torch.cuda.is_available():
    print('Allocated VRAM GB:', round(torch.cuda.memory_allocated() / 1024**3, 2))

In [ ]:
# Cell 5: Import MeKi modules
from src.models.meki_hybrid import MeKiHybridInjector, MeKiBenchmark
print('MeKi module imported successfully')

In [ ]:
# Cell 6 (IMPORTANT): Hybrid Injection Setup
injector = MeKiHybridInjector(model, tokenizer, max_entries=4096, top_k=8)
benchmark = MeKiBenchmark(injector)
print('Hybrid injector ready')
print('Initial stats:', injector.stats())

In [ ]:
# Cell 7: Quick benchmark questions
math_questions = [
    'What is 5 + 3?', 'Calculate 10 times 2.', 'What is 15 minus 7?',
    'What is 8 divided by 2?', 'Calculate 3 plus 3.', 'What is 100 plus 200?',
    'Calculate 7 times 8.', 'What is 50 minus 25?', 'What is 17 plus 29?',
    'Calculate 6 times 7.'
]
mixed_questions = [
    'What is 5 + 3?', 'What is photosynthesis?', 'Calculate 6 times 7.',
    'What is DNA?', 'What is 100 plus 200?'
]
print('Question sets ready')

In [ ]:
# Cell 8: Run benchmark
benchmark.run_phase1_learning(math_questions)
benchmark.run_phase2_recall(mixed_questions)
report = benchmark.report()
report

In [ ]:
# Cell 9: Save report
import json, os
os.makedirs('results', exist_ok=True)
with open('results/meki_benchmark.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, default=str)
print('Saved: results/meki_benchmark.json')